<a href="https://www.kaggle.com/code/beyzadurduu/xgboots-credit-card-fraud-detect?scriptVersionId=354940549" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mlg-ulb/creditcardfraud/creditcard.csv


In [2]:
df=pd.read_csv("/kaggle/input/datasets/mlg-ulb/creditcardfraud/creditcard.csv")

In [3]:
X = df.drop('Class', axis=1)
y = df['Class']

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
import numpy as np

def get_score(n_est, X, y):
    # Pipeline oluşturma
    my_pipeline = Pipeline(steps=[
        ('model', xgb.XGBClassifier(
            n_estimators=n_est,          # Fonksiyondan gelen dinamik değer
            max_depth=5,
            learning_rate=0.05,
            eval_metric='aucpr',
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    # Dengesiz veri seti için Stratified K-Fold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Cross-validation ile ortalama AUPRC skorunu hesaplama
    # (Not: Scikit-learn 'average_precision' metriğini kullanır, yüksek olan iyidir)
    scores = cross_val_score(my_pipeline, X, y, cv=cv, scoring='average_precision', n_jobs=-1)
    
    return scores.mean()

# 100 ile 500 arasında 50'şer adımlarla artan n_estimators değerlerini test etme
estimators_list = range(100, 501, 50)

best_n = None
best_score = -1
results = {}

print("Farklı n_estimators değerleri test ediliyor...\n")

for n in estimators_list:
    # Fonksiyonu çağırırken X ve y verilerini de göndermeliyiz
    mean_score = get_score(n, X, y)
    results[n] = mean_score
    print(f"n_estimators: {n} -> Ortalama AUPRC Skoru: {mean_score:.4f}")
    
    # En yüksek skoru veren n_estimators değerini kaydetme
    if mean_score > best_score:
        best_score = mean_score
        best_n = n

print(f"\nEn iyi sonuç veren n_estimators değeri: {best_n} (Skor: {best_score:.4f})")

Farklı n_estimators değerleri test ediliyor...

n_estimators: 100 -> Ortalama AUPRC Skoru: 0.8513
n_estimators: 150 -> Ortalama AUPRC Skoru: 0.8561
n_estimators: 200 -> Ortalama AUPRC Skoru: 0.8577
n_estimators: 250 -> Ortalama AUPRC Skoru: 0.8598
n_estimators: 300 -> Ortalama AUPRC Skoru: 0.8597
n_estimators: 350 -> Ortalama AUPRC Skoru: 0.8608
n_estimators: 400 -> Ortalama AUPRC Skoru: 0.8607
n_estimators: 450 -> Ortalama AUPRC Skoru: 0.8606
n_estimators: 500 -> Ortalama AUPRC Skoru: 0.8607

En iyi sonuç veren n_estimators değeri: 350 (Skor: 0.8608)


In [6]:

model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    eval_metric='aucpr',
    random_state=42,
    #early_stopping_rounds=5,
    n_jobs=-1
)

In [7]:
model.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [8]:
from sklearn.metrics import accuracy_score
predict=model.predict(X_test)
acc=accuracy_score(predict,y_test)
print(acc)

0.9995786664794073
